# Task 2 — Clickbait Spoiler Generation

The objective of Task 2 is to generate a spoiler for each clickbait post using the linked article content.

This notebook will begin with simple extractive baselines and then compare them with stronger generation approaches using the METEOR evaluation metric.

In [7]:
import os
import pandas as pd

DATA_DIR = "/kaggle/input/competitions/task-2-clickbait-detection-mse-641-s-26"

train_df = pd.read_json(
    os.path.join(DATA_DIR, "train.jsonl"),
    lines=True
)

val_df = pd.read_json(
    os.path.join(DATA_DIR, "val.jsonl"),
    lines=True
)

test_df = pd.read_json(
    os.path.join(DATA_DIR, "test.jsonl"),
    lines=True
)

sample_solution = pd.read_csv(
    os.path.join(DATA_DIR, "sample_solution.csv")
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("Sample solution shape:", sample_solution.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(val_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

print("\nSample solution columns:")
print(sample_solution.columns.tolist())

Train shape: (3200, 14)
Validation shape: (400, 14)
Test shape: (400, 10)
Sample solution shape: (400, 2)

Train columns:
['uuid', 'postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags']

Validation columns:
['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags', 'id']

Test columns:
['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'id']

Sample solution columns:
['id', 'spoiler']


Before we move on, inspect some of the structure and see how they are represented

In [8]:
preview_columns = [
    "postText",
    "targetTitle",
    "targetParagraphs",
    "spoiler",
    "spoilerPositions",
    "tags"
]

display(train_df[preview_columns].head(5))

print("Spoiler type distribution:")
print(train_df["tags"].astype(str).value_counts())

print("\nExample spoiler object:")
print(train_df.loc[0, "spoiler"])

print("\nExample spoiler positions:")
print(train_df.loc[0, "spoilerPositions"])

print("\nNumber of target paragraphs in first example:")
print(len(train_df.loc[0, "targetParagraphs"]))

,postText,targetTitle,targetParagraphs,spoiler,spoilerPositions,tags
0,"[Wes Welker Wanted Dinner With Tom Brady, But ...","Wes Welker Wanted Dinner With Tom Brady, But P...",[It’ll be just like old times this weekend for...,[how about that morning we go throw?],"[[[3, 151], [3, 186]]]",[passage]
1,[NASA sets date for full recovery of ozone hole],Hole In Ozone Layer Expected To Make Full Reco...,[2070 is shaping up to be a great year for Mot...,[2070],"[[[0, 0], [0, 4]]]",[phrase]
2,[This is what makes employees happy -- and it'...,Intellectual Stimulation Trumps Money For Empl...,"[Despite common belief, money isn't the key to...",[intellectual stimulation],"[[[1, 186], [1, 210]]]",[phrase]
3,[Passion is overrated — 7 work habits you need...,"‘Follow your passion’ is wrong, here are 7 hab...","[It’s common wisdom. Near gospel really, and n...",[Purpose connects us to something bigger and i...,"[[[11, 25], [11, 101]], [[17, 56], [17, 85]], ...",[multi]
4,[The perfect way to cook rice so that it's per...,Revealed: The perfect way to cook rice so that...,"[Boiling rice may seem simple, but there is a ...",[in a rice cooker],"[[[5, 60], [5, 76]]]",[phrase]


Spoiler type distribution:
tags
['phrase']     1367
['passage']    1274
['multi']       559
Name: count, dtype: int64

Example spoiler object:
['how about that morning we go throw?']

Example spoiler positions:
[[[3, 151], [3, 186]]]

Number of target paragraphs in first example:
7


In [10]:
def extract_spoiler_from_positions(target_paragraphs, spoiler_positions):
    extracted_parts = []

    for position in spoiler_positions:
        start_position, end_position = position

        start_paragraph, start_character = start_position
        end_paragraph, end_character = end_position

        if start_paragraph == end_paragraph:
            paragraph = target_paragraphs[start_paragraph]
            extracted_text = paragraph[start_character:end_character]
            extracted_parts.append(extracted_text)

        else:
            parts = []

            parts.append(
                target_paragraphs[start_paragraph][start_character:]
            )

            for paragraph_index in range(
                start_paragraph + 1,
                end_paragraph
            ):
                parts.append(
                    target_paragraphs[paragraph_index]
                )

            parts.append(
                target_paragraphs[end_paragraph][:end_character]
            )

            extracted_parts.append(" ".join(parts))

    return extracted_parts

In [11]:
row_index = 3

reference_spoiler = train_df.loc[row_index, "spoiler"]

reconstructed_spoiler = extract_spoiler_from_positions(
    train_df.loc[row_index, "targetParagraphs"],
    train_df.loc[row_index, "spoilerPositions"]
)

print("Reference:")
print(reference_spoiler)

print("\nReconstructed:")
print(reconstructed_spoiler)

print("\nExact match:", reference_spoiler == reconstructed_spoiler)

Reference:
['Purpose connects us to something bigger and in doing so makes us right sized', 'be ruthless with your "No’s."', 'Practice means greatness is doable ... one tiny step after another', 'planning of the SMART goal and number-crunching variety', 'Objectivity — the ability to see the world as it truly is']

Reconstructed:
['Purpose connects us to something bigger and in doing so makes us right sized', 'be ruthless with your "No’s."', 'Practice means greatness is doable ... one tiny step after another', 'planning of the SMART goal and number-crunching variety', 'Objectivity — the ability to see the world as it truly is']

Exact match: True


1. TF-IDF Extractive Baseline

The first baseline selects one sentence from the linked article.

For each example:

1. The article paragraphs are divided into sentences.
2. TF-IDF vectors are created for the clickbait post and candidate sentences.
3. Cosine similarity is calculated.
4. The sentence most similar to the clickbait post is selected as the predicted spoiler.

This provides a simple extractive benchmark before testing more advanced models.

In [12]:
import re
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def clean_text(value):
    if value is None:
        return ""

    if isinstance(value, list):
        return " ".join(str(item) for item in value)

    return str(value)


def split_into_sentences(paragraphs):
    sentences = []

    for paragraph in paragraphs:
        paragraph = str(paragraph).strip()

        if not paragraph:
            continue

        paragraph_sentences = re.split(
            r"(?<=[.!?])\s+",
            paragraph
        )

        sentences.extend(
            sentence.strip()
            for sentence in paragraph_sentences
            if sentence.strip()
        )

    return sentences


def tfidf_extract_spoiler(post_text, target_paragraphs):
    query = clean_text(post_text)
    candidate_sentences = split_into_sentences(
        target_paragraphs
    )

    if not candidate_sentences:
        return ""

    documents = [query] + candidate_sentences

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2)
    )

    try:
        tfidf_matrix = vectorizer.fit_transform(documents)
    except ValueError:
        return candidate_sentences[0]

    query_vector = tfidf_matrix[0]
    sentence_vectors = tfidf_matrix[1:]

    similarities = cosine_similarity(
        query_vector,
        sentence_vectors
    ).flatten()

    best_sentence_index = int(
        np.argmax(similarities)
    )

    return candidate_sentences[best_sentence_index]

In [13]:
for row_index in range(5):
    predicted_spoiler = tfidf_extract_spoiler(
        val_df.loc[row_index, "postText"],
        val_df.loc[row_index, "targetParagraphs"]
    )

    reference_spoiler = " ".join(
        val_df.loc[row_index, "spoiler"]
    )

    print(f"Example {row_index}")
    print("Post:", clean_text(val_df.loc[row_index, "postText"]))
    print("Reference:", reference_spoiler)
    print("Prediction:", predicted_spoiler)
    print("-" * 100)

Example 0
Post: Five Nights at Freddy’s Sequel Delayed for Weird Reason
Reference: some of the plot elements are so disturbing that they are making him feel sick
Prediction: Cawthon’s reason for suddenly delaying Five Nights at Freddy’s Sister Location from its planned October 7th release date doesn’t make much sense.
----------------------------------------------------------------------------------------------------
Example 1
Post: Why Arizona Sheriff Joe Arpaio’s fate could hang on a single word
Reference: "intentionally" could transform a court case against Phoenix-area Sheriff Joe Arpaio from civil charges to a criminal prosecution
Prediction: PHOENIX — A single word — "intentionally" — could transform a court case against Phoenix-area Sheriff Joe Arpaio from civil charges to a criminal prosecution.
----------------------------------------------------------------------------------------------------
Example 2
Post: Here’s how much you should be tipping your hairdresser
Reference: 20

## 2. Evaluate the TF-IDF Baseline

The TF-IDF baseline is evaluated on the validation set using METEOR

In [14]:
import nltk
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet")
nltk.download("omw-1.4")


def spoiler_list_to_text(value):
    if isinstance(value, list):
        return " ".join(str(item) for item in value)

    return str(value)


tfidf_val_predictions = []

for _, row in val_df.iterrows():
    prediction = tfidf_extract_spoiler(
        row["postText"],
        row["targetParagraphs"]
    )

    tfidf_val_predictions.append(prediction)


tfidf_meteor_scores = []

for reference, prediction in zip(
    val_df["spoiler"],
    tfidf_val_predictions
):
    reference_text = spoiler_list_to_text(reference)

    score = meteor_score(
        [reference_text.split()],
        prediction.split()
    )

    tfidf_meteor_scores.append(score)


TFIDF_METEOR = float(np.mean(tfidf_meteor_scores))

print("TF-IDF validation METEOR:", TFIDF_METEOR)
print("Number of predictions:", len(tfidf_val_predictions))

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


TF-IDF validation METEOR: 0.12463984128886132
Number of predictions: 400


## 3. Semantic Extractive Baseline

The TF-IDF baseline achieved a validation METEOR score of 0.1246.

Its main limitation is that it relies heavily on exact word overlap. This experiment uses sentence embeddings to compare the meaning of the clickbait post with each candidate sentence in the linked article.

The candidate sentence with the highest cosine similarity is selected as the predicted spoiler.

In [15]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Sentence embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence embedding model loaded.


In [16]:
def semantic_extract_spoiler(
    post_text,
    target_title,
    target_paragraphs
):
    post = clean_text(post_text)
    title = clean_text(target_title)

    # Include the title to give the query more context
    query = f"{post} {title}".strip()

    candidate_sentences = split_into_sentences(
        target_paragraphs
    )

    if not candidate_sentences:
        return ""

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    candidate_embeddings = embedding_model.encode(
        candidate_sentences,
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        query_embedding,
        candidate_embeddings
    )[0]

    best_index = int(np.argmax(similarities))

    return candidate_sentences[best_index]

In [17]:
for row_index in range(5):
    prediction = semantic_extract_spoiler(
        val_df.loc[row_index, "postText"],
        val_df.loc[row_index, "targetTitle"],
        val_df.loc[row_index, "targetParagraphs"]
    )

    reference = spoiler_list_to_text(
        val_df.loc[row_index, "spoiler"]
    )

    print(f"Example {row_index}")
    print("Post:", clean_text(val_df.loc[row_index, "postText"]))
    print("Reference:", reference)
    print("Prediction:", prediction)
    print("-" * 100)

Example 0
Post: Five Nights at Freddy’s Sequel Delayed for Weird Reason
Reference: some of the plot elements are so disturbing that they are making him feel sick
Prediction: Five Nights at Freddy’s creator Scott Cawthon takes to Steam to tease a possible delay for Five Nights at Freddy’s: Sister Location, the fifth game in the series.
----------------------------------------------------------------------------------------------------
Example 1
Post: Why Arizona Sheriff Joe Arpaio’s fate could hang on a single word
Reference: "intentionally" could transform a court case against Phoenix-area Sheriff Joe Arpaio from civil charges to a criminal prosecution
Prediction: On May 31, Snow will determine the civil penalties and examine whether Arpaio, the sheriff of Arizona’s Maricopa County, will be referred to Arizona’s U.S.
----------------------------------------------------------------------------------------------------
Example 2
Post: Here’s how much you should be tipping your hairdresser

In [18]:
semantic_val_predictions = []

for _, row in tqdm(
    val_df.iterrows(),
    total=len(val_df)
):
    prediction = semantic_extract_spoiler(
        row["postText"],
        row["targetTitle"],
        row["targetParagraphs"]
    )

    semantic_val_predictions.append(prediction)


semantic_meteor_scores = []

for reference, prediction in zip(
    val_df["spoiler"],
    semantic_val_predictions
):
    reference_text = spoiler_list_to_text(reference)

    score = meteor_score(
        [reference_text.split()],
        prediction.split()
    )

    semantic_meteor_scores.append(score)


SEMANTIC_METEOR = float(
    np.mean(semantic_meteor_scores)
)

print("TF-IDF validation METEOR:", TFIDF_METEOR)
print("Semantic validation METEOR:", SEMANTIC_METEOR)
print("Difference:", SEMANTIC_METEOR - TFIDF_METEOR)

  0%|          | 0/400 [00:00<?, ?it/s]

TF-IDF validation METEOR: 0.12463984128886132
Semantic validation METEOR: 0.15855674866454395
Difference: 0.03391690737568262


## 4. Type-Aware Semantic Extraction

The previous semantic model always selected one sentence. However, the dataset contains three spoiler types:

- `phrase`
- `passage`
- `multi`

This experiment adapts the extracted output length to the known validation spoiler type. It is used to test whether type-aware extraction can improve METEOR before building a test-time type prediction strategy.

In [19]:
def get_spoiler_type(value):
    if isinstance(value, list) and len(value) > 0:
        return str(value[0])

    return str(value)


def semantic_extract_type_aware(
    post_text,
    target_title,
    target_paragraphs,
    spoiler_type
):
    post = clean_text(post_text)
    title = clean_text(target_title)
    query = f"{post} {title}".strip()

    candidate_sentences = split_into_sentences(
        target_paragraphs
    )

    if not candidate_sentences:
        return ""

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    candidate_embeddings = embedding_model.encode(
        candidate_sentences,
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        query_embedding,
        candidate_embeddings
    )[0]

    ranked_indices = np.argsort(similarities)[::-1]
    spoiler_type = get_spoiler_type(spoiler_type)

    if spoiler_type == "phrase":
        return candidate_sentences[int(ranked_indices[0])]

    if spoiler_type == "passage":
        best_index = int(ranked_indices[0])

        selected_indices = [
            index
            for index in [best_index, best_index + 1]
            if index < len(candidate_sentences)
        ]

        return " ".join(
            candidate_sentences[index]
            for index in selected_indices
        )

    # multi
    selected_indices = sorted(
        int(index)
        for index in ranked_indices[:3]
    )

    return " ".join(
        candidate_sentences[index]
        for index in selected_indices
    )

In [20]:
type_aware_val_predictions = []

for _, row in tqdm(
    val_df.iterrows(),
    total=len(val_df)
):
    prediction = semantic_extract_type_aware(
        row["postText"],
        row["targetTitle"],
        row["targetParagraphs"],
        row["tags"]
    )

    type_aware_val_predictions.append(prediction)


type_aware_meteor_scores = []

for reference, prediction in zip(
    val_df["spoiler"],
    type_aware_val_predictions
):
    reference_text = spoiler_list_to_text(reference)

    score = meteor_score(
        [reference_text.split()],
        prediction.split()
    )

    type_aware_meteor_scores.append(score)


TYPE_AWARE_METEOR = float(
    np.mean(type_aware_meteor_scores)
)

print("TF-IDF validation METEOR:", TFIDF_METEOR)
print("Semantic validation METEOR:", SEMANTIC_METEOR)
print("Type-aware semantic METEOR:", TYPE_AWARE_METEOR)

  0%|          | 0/400 [00:00<?, ?it/s]

TF-IDF validation METEOR: 0.12463984128886132
Semantic validation METEOR: 0.15855674866454395
Type-aware semantic METEOR: 0.20785911436598895


## 5. Prepare Spoiler-Type Classifier Data

The type-aware extraction experiment used the true spoiler types and achieved a validation METEOR score of 0.2079.

Since spoiler types are unavailable for the test set, a RoBERTa classifier is prepared to predict whether each example requires a `phrase`, `passage`, or `multi` spoiler.

The classifier uses the clickbait post together with the article paragraphs, following the strongest Task 1 configuration.

In [21]:
import random
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

CLASSIFIER_MODEL_NAME = "roberta-base"
CLASSIFIER_MAX_LENGTH = 256
CLASSIFIER_SEED = 42

label2id = {
    "phrase": 0,
    "passage": 1,
    "multi": 2
}

id2label = {
    0: "phrase",
    1: "passage",
    2: "multi"
}

In [22]:
def prepare_classifier_text(row):
    post = clean_text(row["postText"])
    paragraphs = clean_text(row["targetParagraphs"])

    return f"{post} </s></s> {paragraphs}"


for dataframe in [train_df, val_df, test_df]:
    dataframe["classifier_text"] = dataframe.apply(
        prepare_classifier_text,
        axis=1
    )


train_df["label"] = train_df["tags"].apply(
    lambda value: label2id[get_spoiler_type(value)]
)

val_df["label"] = val_df["tags"].apply(
    lambda value: label2id[get_spoiler_type(value)]
)


print(
    train_df[
        ["classifier_text", "tags", "label"]
    ].head()
)

print("\nTraining label distribution:")
print(train_df["label"].value_counts().sort_index())

                                     classifier_text       tags  label
0  Wes Welker Wanted Dinner With Tom Brady, But P...  [passage]      1
1  NASA sets date for full recovery of ozone hole...   [phrase]      0
2  This is what makes employees happy -- and it's...   [phrase]      0
3  Passion is overrated — 7 work habits you need ...    [multi]      2
4  The perfect way to cook rice so that it's perf...   [phrase]      0

Training label distribution:
label
0    1367
1    1274
2     559
Name: count, dtype: int64


In [23]:
classifier_tokenizer = AutoTokenizer.from_pretrained(
    CLASSIFIER_MODEL_NAME
)

classifier_train_dataset = Dataset.from_pandas(
    train_df[
        ["classifier_text", "label"]
    ].reset_index(drop=True)
)

classifier_val_dataset = Dataset.from_pandas(
    val_df[
        ["classifier_text", "label"]
    ].reset_index(drop=True)
)

classifier_test_dataset = Dataset.from_pandas(
    test_df[
        ["classifier_text"]
    ].reset_index(drop=True)
)


def tokenize_classifier(batch):
    return classifier_tokenizer(
        batch["classifier_text"],
        truncation=True,
        max_length=CLASSIFIER_MAX_LENGTH
    )


classifier_train_dataset = classifier_train_dataset.map(
    tokenize_classifier,
    batched=True,
    remove_columns=["classifier_text"]
)

classifier_val_dataset = classifier_val_dataset.map(
    tokenize_classifier,
    batched=True,
    remove_columns=["classifier_text"]
)

classifier_test_dataset = classifier_test_dataset.map(
    tokenize_classifier,
    batched=True,
    remove_columns=["classifier_text"]
)


print(classifier_train_dataset)
print(classifier_val_dataset)
print(classifier_test_dataset)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 3200
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 400
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 400
})


## 6. Train the Spoiler-Type Classifier

A RoBERTa-base sequence classifier is fine-tuned to predict the spoiler type.

The classifier is evaluated using macro F1 because the three classes are imbalanced and all spoiler types should contribute equally to the evaluation.

In [24]:
def compute_classifier_metrics(eval_prediction):
    logits, labels = eval_prediction

    predictions = np.argmax(
        logits,
        axis=1
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro"
        ),
        "weighted_f1": f1_score(
            labels,
            predictions,
            average="weighted"
        )
    }


set_seed(CLASSIFIER_SEED)

classifier_model = AutoModelForSequenceClassification.from_pretrained(
    CLASSIFIER_MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

print("Classifier model loaded.")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Classifier model loaded.


In [25]:
classifier_training_args = TrainingArguments(
    output_dir="/kaggle/working/task2_type_classifier",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=4,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=1,
    report_to="none",

    seed=CLASSIFIER_SEED,
    fp16=torch.cuda.is_available()
)


classifier_trainer = Trainer(
    model=classifier_model,
    args=classifier_training_args,

    train_dataset=classifier_train_dataset,
    eval_dataset=classifier_val_dataset,

    processing_class=classifier_tokenizer,
    compute_metrics=compute_classifier_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

In [26]:
classifier_train_result = classifier_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.986382,1.706146,0.632500,0.622159,0.618255
2,1.504154,1.338970,0.730000,0.710527,0.724323
3,1.187262,1.344425,0.757500,0.749236,0.756050
4,0.904393,1.359070,0.765000,0.759374,0.764142


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [27]:
classifier_validation_results = classifier_trainer.evaluate()

classifier_validation_results

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.3590704202651978,
 'eval_accuracy': 0.765,
 'eval_macro_f1': 0.7593739888299819,
 'eval_weighted_f1': 0.7641418921635641,
 'eval_runtime': 3.9616,
 'eval_samples_per_second': 100.97,
 'eval_steps_per_second': 3.282,
 'epoch': 4.0}

In [28]:
classifier_val_output = classifier_trainer.predict(
    classifier_val_dataset
)

classifier_val_logits = classifier_val_output.predictions

classifier_val_label_ids = np.argmax(
    classifier_val_logits,
    axis=1
)

classifier_val_predicted_types = [
    id2label[int(label_id)]
    for label_id in classifier_val_label_ids
]

print(
    pd.Series(
        classifier_val_predicted_types
    ).value_counts()
)

print("\nFirst 10 predicted types:")
print(classifier_val_predicted_types[:10])

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


passage    169
phrase     168
multi       63
Name: count, dtype: int64

First 10 predicted types:
['passage', 'passage', 'phrase', 'phrase', 'passage', 'phrase', 'phrase', 'passage', 'passage', 'passage']


In [29]:
predicted_type_val_predictions = []

for row_position, (_, row) in enumerate(
    tqdm(
        val_df.iterrows(),
        total=len(val_df)
    )
):
    prediction = semantic_extract_type_aware(
        row["postText"],
        row["targetTitle"],
        row["targetParagraphs"],
        classifier_val_predicted_types[row_position]
    )

    predicted_type_val_predictions.append(
        prediction
    )


predicted_type_meteor_scores = []

for reference, prediction in zip(
    val_df["spoiler"],
    predicted_type_val_predictions
):
    reference_text = spoiler_list_to_text(
        reference
    )

    score = meteor_score(
        [reference_text.split()],
        prediction.split()
    )

    predicted_type_meteor_scores.append(
        score
    )


PREDICTED_TYPE_METEOR = float(
    np.mean(predicted_type_meteor_scores)
)

print("Single-sentence semantic METEOR:", SEMANTIC_METEOR)
print("True-type semantic METEOR:", TYPE_AWARE_METEOR)
print("Predicted-type semantic METEOR:", PREDICTED_TYPE_METEOR)

  0%|          | 0/400 [00:00<?, ?it/s]

Single-sentence semantic METEOR: 0.15855674866454395
True-type semantic METEOR: 0.20785911436598895
Predicted-type semantic METEOR: 0.20499894609238228


In [30]:
classifier_test_output = classifier_trainer.predict(
    classifier_test_dataset
)

classifier_test_logits = classifier_test_output.predictions

classifier_test_label_ids = np.argmax(
    classifier_test_logits,
    axis=1
)

classifier_test_predicted_types = [
    id2label[int(label_id)]
    for label_id in classifier_test_label_ids
]

print("Predicted test type distribution:")
print(
    pd.Series(
        classifier_test_predicted_types
    ).value_counts()
)

print("\nFirst 10 predicted test types:")
print(classifier_test_predicted_types[:10])

Predicted test type distribution:
passage    188
phrase     160
multi       52
Name: count, dtype: int64

First 10 predicted test types:
['phrase', 'passage', 'phrase', 'phrase', 'passage', 'phrase', 'multi', 'passage', 'phrase', 'multi']


In [31]:
test_spoiler_predictions = []

for row_position, (_, row) in enumerate(
    tqdm(
        test_df.iterrows(),
        total=len(test_df)
    )
):
    prediction = semantic_extract_type_aware(
        row["postText"],
        row["targetTitle"],
        row["targetParagraphs"],
        classifier_test_predicted_types[row_position]
    )

    test_spoiler_predictions.append(
        prediction
    )

print("Number of test predictions:", len(test_spoiler_predictions))
print("\nFirst 5 predictions:")

for prediction in test_spoiler_predictions[:5]:
    print(prediction)
    print("-" * 80)

  0%|          | 0/400 [00:00<?, ?it/s]

Number of test predictions: 400

First 5 predictions:
He has balloons and a sign in hand that reads, "Heard urine need of a kidney, want mine?" It seems like the young man is quite the jokester!
--------------------------------------------------------------------------------
And in the workplace putting your needs before those of your colleagues is often seen as selfish behaviour. But new research says being selfless at work can backfire.
--------------------------------------------------------------------------------
But the ability of money and the things it buys—access to better medical care, leisure time, healthier food—to stave off death is well known.
--------------------------------------------------------------------------------
In other news, I’ve literally never won a game of Scrabble in my life.
--------------------------------------------------------------------------------
Even better if you're raising chickens at home and have your own fresh eggs on hand. Fortunately, the

In [32]:
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "spoiler": test_spoiler_predictions
})

print(submission_df.head())
print("\nShape:", submission_df.shape)
print("\nMissing spoilers:", submission_df["spoiler"].isna().sum())
print("Empty spoilers:", (submission_df["spoiler"].str.strip() == "").sum())

   id                                            spoiler
0   0  He has balloons and a sign in hand that reads,...
1   1  And in the workplace putting your needs before...
2   2  But the ability of money and the things it buy...
3   3  In other news, I’ve literally never won a game...
4   4  Even better if you're raising chickens at home...

Shape: (400, 2)

Missing spoilers: 0
Empty spoilers: 0


In [33]:
submission_path = "/kaggle/working/task2_semantic_type_aware_submission.csv"

submission_df.to_csv(
    submission_path,
    index=False
)

print("Saved to:", submission_path)

Saved to: /kaggle/working/task2_semantic_type_aware_submission.csv


## 7. Prepare FLAN-T5 Generation Data

The extractive semantic pipeline achieved a Kaggle METEOR score of 0.2112.

Its main limitation is that it selects complete article sentences and cannot reliably produce short answer spans or combine multiple spoiler elements.

A FLAN-T5 sequence-to-sequence model is prepared to generate the spoiler directly from the spoiler type, clickbait post, article title, and article paragraphs.

In [34]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

GENERATION_MODEL_NAME = "google/flan-t5-base"

GENERATION_MAX_INPUT_LENGTH = 512
GENERATION_MAX_TARGET_LENGTH = 128

In [35]:
def prepare_generation_input(row, spoiler_type):
    post = clean_text(row["postText"])
    title = clean_text(row["targetTitle"])
    paragraphs = clean_text(row["targetParagraphs"])

    return (
        f"Generate a {spoiler_type} spoiler. "
        f"Clickbait post: {post} "
        f"Article title: {title} "
        f"Article: {paragraphs}"
    )


def prepare_generation_target(spoiler):
    return spoiler_list_to_text(spoiler)

In [36]:
generation_train_df = pd.DataFrame({
    "input_text": train_df.apply(
        lambda row: prepare_generation_input(
            row,
            get_spoiler_type(row["tags"])
        ),
        axis=1
    ),
    "target_text": train_df["spoiler"].apply(
        prepare_generation_target
    )
})


generation_val_df = pd.DataFrame({
    "input_text": val_df.apply(
        lambda row: prepare_generation_input(
            row,
            get_spoiler_type(row["tags"])
        ),
        axis=1
    ),
    "target_text": val_df["spoiler"].apply(
        prepare_generation_target
    )
})


print("Training shape:", generation_train_df.shape)
print("Validation shape:", generation_val_df.shape)

print("\nExample input:")
print(generation_train_df.loc[0, "input_text"][:1000])

print("\nExample target:")
print(generation_train_df.loc[0, "target_text"])

Training shape: (3200, 2)
Validation shape: (400, 2)

Example input:
Generate a passage spoiler. Clickbait post: Wes Welker Wanted Dinner With Tom Brady, But Patriots QB Had Better Idea Article title: Wes Welker Wanted Dinner With Tom Brady, But Patriots QB Had A Better Idea Article: It’ll be just like old times this weekend for Tom Brady and Wes Welker. Welker revealed Friday morning on a Miami radio station that he contacted Brady because he’ll be in town for Sunday’s game between the New England Patriots and Miami Dolphins at Gillette Stadium. It seemed like a perfect opportunity for the two to catch up. But Brady’s definition of "catching up" involves far more than just a meal. In fact, it involves some literal "catching" as the Patriots quarterback looks to stay sharp during his four-game Deflategate suspension. "I hit him up to do dinner Saturday night. He’s like, ‘I’m going to be flying in from Ann Arbor later (after the Michigan-Colorado football game), but how about that morni

In [37]:
generation_tokenizer = AutoTokenizer.from_pretrained(
    GENERATION_MODEL_NAME
)

generation_model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATION_MODEL_NAME
)


generation_train_dataset = Dataset.from_pandas(
    generation_train_df.reset_index(drop=True)
)

generation_val_dataset = Dataset.from_pandas(
    generation_val_df.reset_index(drop=True)
)


def tokenize_generation(batch):
    model_inputs = generation_tokenizer(
        batch["input_text"],
        max_length=GENERATION_MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = generation_tokenizer(
        text_target=batch["target_text"],
        max_length=GENERATION_MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


generation_train_dataset = generation_train_dataset.map(
    tokenize_generation,
    batched=True,
    remove_columns=[
        "input_text",
        "target_text"
    ]
)

generation_val_dataset = generation_val_dataset.map(
    tokenize_generation,
    batched=True,
    remove_columns=[
        "input_text",
        "target_text"
    ]
)


generation_data_collator = DataCollatorForSeq2Seq(
    tokenizer=generation_tokenizer,
    model=generation_model
)

print(generation_train_dataset)
print(generation_val_dataset)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3200
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 400
})


## 8. Train FLAN-T5

The FLAN-T5 model is fine-tuned to generate spoiler text directly from the spoiler type, clickbait post, article title, and article content.

The best checkpoint is selected using validation loss.

In [38]:
generation_training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/task2_flan_t5_base",

    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,

    num_train_epochs=3,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    predict_with_generate=True,
    generation_max_length=GENERATION_MAX_TARGET_LENGTH,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=1,
    report_to="none",

    fp16=torch.cuda.is_available(),
    seed=42
)

In [39]:
generation_trainer = Seq2SeqTrainer(
    model=generation_model,
    args=generation_training_args,

    train_dataset=generation_train_dataset,
    eval_dataset=generation_val_dataset,

    processing_class=generation_tokenizer,
    data_collator=generation_data_collator
)

In [40]:
generation_train_result = generation_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,16.038350,13.263091
2,14.134716,12.284228
3,13.557058,12.019958


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


In [41]:
generation_eval_results = generation_trainer.evaluate()

generation_eval_results

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 12.01995849609375,
 'eval_runtime': 17.4047,
 'eval_samples_per_second': 22.982,
 'eval_steps_per_second': 1.436,
 'epoch': 3.0}

In [42]:
generation_val_output = generation_trainer.predict(
    generation_val_dataset
)

generation_val_token_ids = generation_val_output.predictions

# Some Trainer versions return a tuple
if isinstance(generation_val_token_ids, tuple):
    generation_val_token_ids = generation_val_token_ids[0]

flan_val_predictions = generation_tokenizer.batch_decode(
    generation_val_token_ids,
    skip_special_tokens=True
)

print("Number of predictions:", len(flan_val_predictions))

for row_index in range(5):
    print(f"\nExample {row_index}")
    print("Post:", clean_text(val_df.loc[row_index, "postText"]))
    print(
        "Reference:",
        spoiler_list_to_text(val_df.loc[row_index, "spoiler"])
    )
    print("Prediction:", flan_val_predictions[row_index])
    print("-" * 100)

Number of predictions: 400

Example 0
Post: Five Nights at Freddy’s Sequel Delayed for Weird Reason
Reference: some of the plot elements are so disturbing that they are making him feel sick
Prediction: a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a 
----------------------------------------------------------------------------------------------------

Example 1
Post: Why Arizona Sheriff Joe Arpaio’s fate could hang on a single word
Reference: "intentionally" could transform a court case against Phoenix-area Sheriff Joe Arpaio from civil charges to a criminal prosecution
Prediction: a a sassy a sassy a sassy a sassy a sassy a sassy a sassy a sassy a sassy a sass a sass a sass a sass a sass a sass a sass a s
----------------------------------------------------------------------------------------------------

Example 2
Post: Here’s how much you should be tipping your hairdresser
Reference: 20%
Prediction: a stylt

In [44]:
flan_meteor_scores = []

for reference, prediction in zip(
    val_df["spoiler"],
    flan_val_predictions
):
    reference_text = spoiler_list_to_text(reference)

    score = meteor_score(
        [reference_text.split()],
        prediction.split()
    )

    flan_meteor_scores.append(score)

FLAN_METEOR = float(np.mean(flan_meteor_scores))

print("Semantic extractive METEOR:", PREDICTED_TYPE_METEOR)
print("FLAN-T5 validation METEOR:", FLAN_METEOR)

print(
    "Empty FLAN-T5 predictions:",
    sum(not prediction.strip() for prediction in flan_val_predictions)
)

Semantic extractive METEOR: 0.20499894609238228
FLAN-T5 validation METEOR: 0.043778874639473414
Empty FLAN-T5 predictions: 0


In [50]:
import gc
import torch

try:
    del generation_trainer
except:
    pass

try:
    del generation_model
except:
    pass

gc.collect()
torch.cuda.empty_cache()

generation_model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATION_MODEL_NAME
)

generation_data_collator = DataCollatorForSeq2Seq(
    tokenizer=generation_tokenizer,
    model=generation_model
)

print("Clean model loaded.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Clean model loaded.


In [51]:
small_generation_train_dataset = (
    generation_train_dataset
    .shuffle(seed=42)
    .select(range(500))
)

small_generation_val_dataset = (
    generation_val_dataset
    .select(range(100))
)

print(small_generation_train_dataset)
print(small_generation_val_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})


In [52]:
diagnostic_training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/task2_flan_t5_diagnostic",

    learning_rate=1e-4,
    optim="adafactor",

    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,

    num_train_epochs=1,
    max_grad_norm=1.0,
    label_smoothing_factor=0.1,

    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",

    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=2,

    report_to="none",
    fp16=False,
    seed=42
)

In [53]:
diagnostic_trainer = Seq2SeqTrainer(
    model=generation_model,
    args=diagnostic_training_args,

    train_dataset=small_generation_train_dataset,
    eval_dataset=small_generation_val_dataset,

    processing_class=generation_tokenizer,
    data_collator=generation_data_collator
)

diagnostic_trainer.train()

Epoch,Training Loss,Validation Loss
1,35.113432,16.034081


TrainOutput(global_step=63, training_loss=35.11343238467262, metrics={'train_runtime': 81.9178, 'train_samples_per_second': 6.104, 'train_steps_per_second': 0.769, 'total_flos': 342378676224000.0, 'train_loss': 35.11343238467262, 'epoch': 1.0})

In [55]:
for row_index in range(len(train_df)):
    positions = train_df.loc[row_index, "spoilerPositions"]

    valid_simple_format = True

    for position in positions:
        if not (
            isinstance(position, list)
            and len(position) == 2
            and isinstance(position[0], list)
            and isinstance(position[1], list)
        ):
            valid_simple_format = False
            break

    if not valid_simple_format:
        print("First unusual row:", row_index)
        print("Tag:", train_df.loc[row_index, "tags"])
        print("Spoiler:", train_df.loc[row_index, "spoiler"])
        print("Positions:", positions)
        print("Paragraph count:", len(train_df.loc[row_index, "targetParagraphs"]))
        break

First unusual row: 546
Tag: ['passage']
Spoiler: ['at level 256, there’s only enough memory in the game for the left-half of the board. The right-half of the game is filled with computer garble that looks something like the Matrix code. "You get to the end, and there’s nothing to do but die."']
Positions: [[[4, 836]]]
Paragraph count: 6


In [56]:
def find_spoiler_in_paragraphs(spoiler_text, paragraphs):
    spoiler_text = str(spoiler_text).strip()

    for paragraph_index, paragraph in enumerate(paragraphs):
        paragraph = str(paragraph)

        start_character = paragraph.find(spoiler_text)

        if start_character != -1:
            end_character = start_character + len(spoiler_text)

            return {
                "paragraph_index": paragraph_index,
                "start_character": start_character,
                "end_character": end_character
            }

    return None


search_results = []

for _, row in train_df.iterrows():
    spoiler_type = get_spoiler_type(row["tags"])
    spoiler_parts = row["spoiler"]

    matched_parts = 0

    for spoiler_part in spoiler_parts:
        match = find_spoiler_in_paragraphs(
            spoiler_part,
            row["targetParagraphs"]
        )

        if match is not None:
            matched_parts += 1

    search_results.append({
        "spoiler_type": spoiler_type,
        "total_parts": len(spoiler_parts),
        "matched_parts": matched_parts,
        "all_parts_matched": matched_parts == len(spoiler_parts)
    })


search_results_df = pd.DataFrame(search_results)

print("Fully matched examples:")
print(
    search_results_df.groupby("spoiler_type")[
        "all_parts_matched"
    ].agg(["sum", "count", "mean"])
)

print("\nTotal spoiler parts:")
print(search_results_df["total_parts"].sum())

print(
    "\nMatched spoiler parts:",
    search_results_df["matched_parts"].sum()
)

print(
    "\nOverall part matching rate:",
    search_results_df["matched_parts"].sum()
    / search_results_df["total_parts"].sum()
)

Fully matched examples:
               sum  count      mean
spoiler_type                       
multi          527    559  0.942755
passage       1222   1274  0.959184
phrase        1321   1367  0.966350

Total spoiler parts:
4622

Matched spoiler parts: 4470

Overall part matching rate: 0.9671138035482475


In [57]:
def build_qa_examples(dataframe):
    qa_examples = []

    for row_index, row in dataframe.iterrows():
        question = (
            clean_text(row["postText"])
            + " "
            + clean_text(row["targetTitle"])
        ).strip()

        spoiler_type = get_spoiler_type(row["tags"])

        for spoiler_part_index, spoiler_part in enumerate(row["spoiler"]):
            match = find_spoiler_in_paragraphs(
                spoiler_part,
                row["targetParagraphs"]
            )

            if match is None:
                continue

            paragraph_index = match["paragraph_index"]
            context = str(
                row["targetParagraphs"][paragraph_index]
            )

            qa_examples.append({
                "source_row": row_index,
                "spoiler_type": spoiler_type,
                "spoiler_part_index": spoiler_part_index,
                "question": question,
                "context": context,
                "answer_text": str(spoiler_part),
                "answer_start": match["start_character"]
            })

    return pd.DataFrame(qa_examples)


qa_train_df = build_qa_examples(train_df)
qa_val_df = build_qa_examples(val_df)

print("QA training shape:", qa_train_df.shape)
print("QA validation shape:", qa_val_df.shape)

print("\nTraining examples by spoiler type:")
print(qa_train_df["spoiler_type"].value_counts())

print("\nExample:")
print("Question:", qa_train_df.loc[0, "question"])
print("Context:", qa_train_df.loc[0, "context"])
print("Answer:", qa_train_df.loc[0, "answer_text"])
print("Answer start:", qa_train_df.loc[0, "answer_start"])

start = qa_train_df.loc[0, "answer_start"]
answer = qa_train_df.loc[0, "answer_text"]

print(
    "Recovered answer:",
    qa_train_df.loc[0, "context"][
        start:start + len(answer)
    ]
)

QA training shape: (4470, 7)
QA validation shape: (597, 7)

Training examples by spoiler type:
spoiler_type
multi      1927
phrase     1321
passage    1222
Name: count, dtype: int64

Example:
Question: Wes Welker Wanted Dinner With Tom Brady, But Patriots QB Had Better Idea Wes Welker Wanted Dinner With Tom Brady, But Patriots QB Had A Better Idea
Context: "I hit him up to do dinner Saturday night. He’s like, ‘I’m going to be flying in from Ann Arbor later (after the Michigan-Colorado football game), but how about that morning we go throw?’ " Welker said on WQAM, per The Boston Globe. "And I’m just sitting there, I’m like, ‘I was just thinking about dinner, but yeah, sure. I’ll get over there early and we can throw a little bit.’ "
Answer: how about that morning we go throw?
Answer start: 151
Recovered answer: how about that morning we go throw?


## 9. Prepare Extractive QA Data

Each matched spoiler part is treated as an extractive question-answering example.

The clickbait post and article title form the question, while the paragraph containing the spoiler forms the context. The model learns to predict the starting and ending token positions of the spoiler text.

In [58]:
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

QA_MODEL_NAME = "deepset/roberta-base-squad2"
QA_MAX_LENGTH = 384
QA_DOCUMENT_STRIDE = 128

qa_tokenizer = AutoTokenizer.from_pretrained(
    QA_MODEL_NAME,
    use_fast=True
)

print("QA tokenizer loaded.")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

QA tokenizer loaded.


In [59]:
def add_qa_answers(dataframe):
    prepared_df = dataframe.copy()

    prepared_df["answers"] = prepared_df.apply(
        lambda row: {
            "text": [row["answer_text"]],
            "answer_start": [int(row["answer_start"])]
        },
        axis=1
    )

    return prepared_df


qa_train_prepared_df = add_qa_answers(qa_train_df)
qa_val_prepared_df = add_qa_answers(qa_val_df)


qa_train_dataset = Dataset.from_pandas(
    qa_train_prepared_df[
        ["question", "context", "answers"]
    ].reset_index(drop=True)
)

qa_val_dataset = Dataset.from_pandas(
    qa_val_prepared_df[
        ["question", "context", "answers"]
    ].reset_index(drop=True)
)

print(qa_train_dataset)
print(qa_val_dataset)

Dataset({
    features: ['question', 'context', 'answers'],
    num_rows: 4470
})
Dataset({
    features: ['question', 'context', 'answers'],
    num_rows: 597
})


In [63]:
def prepare_qa_features(examples):
    questions = [
        question.strip()
        for question in examples["question"]
    ]

    tokenized_examples = qa_tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=QA_MAX_LENGTH,
        stride=QA_DOCUMENT_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized_examples.pop(
        "overflow_to_sample_mapping"
    )

    offset_mapping = tokenized_examples.pop(
        "offset_mapping"
    )

    start_positions = []
    end_positions = []

    for feature_index, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][feature_index]
        sequence_ids = tokenized_examples.sequence_ids(feature_index)

        cls_index = input_ids.index(
            qa_tokenizer.cls_token_id
        )

        sample_index = sample_mapping[feature_index]
        answers = examples["answers"][sample_index]

        answer_start = answers["answer_start"][0]
        answer_end = answer_start + len(
            answers["text"][0]
        )

        context_token_indices = [
            token_index
            for token_index, sequence_id in enumerate(sequence_ids)
            if sequence_id == 1
        ]

        if not context_token_indices:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        context_start = context_token_indices[0]
        context_end = context_token_indices[-1]

        answer_outside_window = (
            offsets[context_start][0] > answer_start
            or offsets[context_end][1] < answer_end
        )

        if answer_outside_window:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_token = context_start

        while (
            start_token <= context_end
            and offsets[start_token][1] <= answer_start
        ):
            start_token += 1

        end_token = context_end

        while (
            end_token >= context_start
            and offsets[end_token][0] >= answer_end
        ):
            end_token -= 1

        if (
            start_token > context_end
            or end_token < context_start
            or start_token > end_token
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            start_positions.append(start_token)
            end_positions.append(end_token)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions

    return tokenized_examples

In [64]:
qa_train_tokenized = qa_train_dataset.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_train_dataset.column_names
)

qa_val_tokenized = qa_val_dataset.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_val_dataset.column_names
)

print(qa_train_tokenized)
print(qa_val_tokenized)

Map:   0%|          | 0/4470 [00:00<?, ? examples/s]

Map:   0%|          | 0/597 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 4494
})
Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 602
})


In [65]:
qa_model = AutoModelForQuestionAnswering.from_pretrained(
    QA_MODEL_NAME
)

qa_data_collator = DefaultDataCollator()

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [66]:
qa_training_args = TrainingArguments(
    output_dir="/kaggle/working/task2_qa_model",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=2,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=1,
    report_to="none",

    fp16=torch.cuda.is_available(),
    seed=42
)

In [67]:
qa_trainer = Trainer(
    model=qa_model,
    args=qa_training_args,

    train_dataset=qa_train_tokenized,
    eval_dataset=qa_val_tokenized,

    processing_class=qa_tokenizer,
    data_collator=qa_data_collator
)

In [68]:
qa_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,1.516013,1.289698
2,1.087132,1.312824


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=562, training_loss=1.3015726978668538, metrics={'train_runtime': 452.0326, 'train_samples_per_second': 19.884, 'train_steps_per_second': 1.243, 'total_flos': 1761401437157376.0, 'train_loss': 1.3015726978668538, 'epoch': 2.0})

In [69]:
qa_eval_results = qa_trainer.evaluate()
qa_eval_results

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.2891103029251099,
 'eval_runtime': 7.8936,
 'eval_samples_per_second': 76.264,
 'eval_steps_per_second': 2.407,
 'epoch': 2.0}

In [70]:
qa_val_predictions = qa_trainer.predict(
    qa_val_tokenized
)

start_logits, end_logits = qa_val_predictions.predictions

predicted_start_positions = np.argmax(
    start_logits,
    axis=1
)

predicted_end_positions = np.argmax(
    end_logits,
    axis=1
)

print("Number of validation features:", len(predicted_start_positions))

Number of validation features: 602


In [72]:
def prepare_qa_validation_features(examples):
    questions = [
        question.strip()
        for question in examples["question"]
    ]

    tokenized_examples = qa_tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=QA_MAX_LENGTH,
        stride=QA_DOCUMENT_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized_examples.pop(
        "overflow_to_sample_mapping"
    )

    example_ids = []

    for feature_index, sample_index in enumerate(sample_mapping):
        example_ids.append(sample_index)

        sequence_ids = tokenized_examples.sequence_ids(
            feature_index
        )

        tokenized_examples["offset_mapping"][feature_index] = [
            offset if sequence_ids[token_index] == 1 else None
            for token_index, offset in enumerate(
                tokenized_examples["offset_mapping"][feature_index]
            )
        ]

    tokenized_examples["example_id"] = example_ids

    return tokenized_examples

In [74]:
from collections import defaultdict

qa_val_model_features = qa_val_features.remove_columns(
    ["example_id", "offset_mapping"]
)

qa_val_output = qa_trainer.predict(
    qa_val_model_features
)

start_logits, end_logits = qa_val_output.predictions

features_per_example = defaultdict(list)

for feature_index, example_id in enumerate(
    qa_val_features["example_id"]
):
    features_per_example[int(example_id)].append(
        feature_index
    )

print("Original QA examples:", len(qa_val_dataset))
print("Tokenized QA features:", len(qa_val_features))

Original QA examples: 597
Tokenized QA features: 602


In [76]:
def recover_best_qa_predictions(
    examples,
    features,
    start_logits,
    end_logits,
    n_best_size=20,
    max_answer_length=128
):
    predictions = []

    for example_index in range(len(examples)):
        context = examples[example_index]["context"]
        candidate_answers = []

        for feature_index in features_per_example[example_index]:
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(
                start_logits[feature_index]
            )[-n_best_size:][::-1]

            end_indexes = np.argsort(
                end_logits[feature_index]
            )[-n_best_size:][::-1]

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offsets)
                        or end_index >= len(offsets)
                    ):
                        continue

                    if (
                        offsets[start_index] is None
                        or offsets[end_index] is None
                    ):
                        continue

                    if end_index < start_index:
                        continue

                    if (
                        end_index - start_index + 1
                        > max_answer_length
                    ):
                        continue

                    start_character = offsets[start_index][0]
                    end_character = offsets[end_index][1]

                    answer_text = context[
                        start_character:end_character
                    ].strip()

                    if not answer_text:
                        continue

                    score = (
                        start_logits[feature_index][start_index]
                        + end_logits[feature_index][end_index]
                    )

                    candidate_answers.append(
                        {
                            "text": answer_text,
                            "score": float(score)
                        }
                    )

        if candidate_answers:
            best_answer = max(
                candidate_answers,
                key=lambda item: item["score"]
            )["text"]
        else:
            best_answer = ""

        predictions.append(best_answer)

    return predictions

In [77]:
qa_span_predictions = recover_best_qa_predictions(
    qa_val_dataset,
    qa_val_features,
    start_logits,
    end_logits
)

print("Recovered predictions:", len(qa_span_predictions))

for row_index in range(5):
    print(f"\nExample {row_index}")
    print(
        "Reference:",
        qa_val_df.loc[row_index, "answer_text"]
    )
    print(
        "Prediction:",
        qa_span_predictions[row_index]
    )
    print("-" * 100)

Recovered predictions: 597

Example 0
Reference: some of the plot elements are so disturbing that they are making him feel sick
Prediction: too dark
----------------------------------------------------------------------------------------------------

Example 1
Reference: "intentionally"
Prediction: "intentionally"
----------------------------------------------------------------------------------------------------

Example 2
Reference: could transform a court case against Phoenix-area Sheriff Joe Arpaio from civil charges to a criminal prosecution
Prediction: "intentionally"
----------------------------------------------------------------------------------------------------

Example 3
Reference: 20%
Prediction: 20%
----------------------------------------------------------------------------------------------------

Example 4
Reference: CBGB
Prediction: CBGB
----------------------------------------------------------------------------------------------------


In [78]:
qa_span_meteor_scores = []

for reference, prediction in zip(
    qa_val_df["answer_text"],
    qa_span_predictions
):
    score = meteor_score(
        [str(reference).split()],
        str(prediction).split()
    )

    qa_span_meteor_scores.append(score)


qa_val_df["qa_prediction"] = qa_span_predictions
qa_val_df["qa_meteor"] = qa_span_meteor_scores


print("Overall QA-part METEOR:")
print(qa_val_df["qa_meteor"].mean())

print("\nQA-part METEOR by spoiler type:")
print(
    qa_val_df.groupby("spoiler_type")["qa_meteor"]
    .agg(["mean", "count"])
)

Overall QA-part METEOR:
0.6736590605342669

QA-part METEOR by spoiler type:
                  mean  count
spoiler_type                 
multi         0.735748    294
passage       0.588301    147
phrase        0.637079    156


In [80]:
def rank_paragraphs_semantically(
    post_text,
    target_title,
    target_paragraphs
):
    query = (
        clean_text(post_text)
        + " "
        + clean_text(target_title)
    ).strip()

    paragraphs = [
        str(paragraph).strip()
        for paragraph in target_paragraphs
        if str(paragraph).strip()
    ]

    if not paragraphs:
        return []

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    paragraph_embeddings = embedding_model.encode(
        paragraphs,
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        query_embedding,
        paragraph_embeddings
    )[0]

    ranked_indices = np.argsort(similarities)[::-1]

    return [
        int(index)
        for index in ranked_indices
    ]

In [81]:
top1_hits = 0
top3_hits = 0
top5_hits = 0
total_matched_parts = 0

for _, row in val_df.iterrows():
    ranked_paragraphs = rank_paragraphs_semantically(
        row["postText"],
        row["targetTitle"],
        row["targetParagraphs"]
    )

    for spoiler_part in row["spoiler"]:
        match = find_spoiler_in_paragraphs(
            spoiler_part,
            row["targetParagraphs"]
        )

        if match is None:
            continue

        true_paragraph_index = match["paragraph_index"]
        total_matched_parts += 1

        if true_paragraph_index in ranked_paragraphs[:1]:
            top1_hits += 1

        if true_paragraph_index in ranked_paragraphs[:3]:
            top3_hits += 1

        if true_paragraph_index in ranked_paragraphs[:5]:
            top5_hits += 1


print("Matched validation spoiler parts:", total_matched_parts)

print(
    "Top-1 paragraph recall:",
    top1_hits / total_matched_parts
)

print(
    "Top-3 paragraph recall:",
    top3_hits / total_matched_parts
)

print(
    "Top-5 paragraph recall:",
    top5_hits / total_matched_parts
)

Matched validation spoiler parts: 597
Top-1 paragraph recall: 0.21273031825795644
Top-3 paragraph recall: 0.44221105527638194
Top-5 paragraph recall: 0.5778894472361809


In [82]:
def extract_qa_candidate(question, context, max_answer_length=128):
    inputs = qa_tokenizer(
        question,
        context,
        truncation="only_second",
        max_length=QA_MAX_LENGTH,
        return_offsets_mapping=True,
        return_tensors="pt"
    )

    offsets = inputs.pop("offset_mapping")[0].tolist()
    sequence_ids = inputs.sequence_ids(0)

    model_inputs = {
        key: value.to(qa_model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = qa_model(**model_inputs)

    start_scores = outputs.start_logits[0].cpu().numpy()
    end_scores = outputs.end_logits[0].cpu().numpy()

    context_indices = [
        index
        for index, sequence_id in enumerate(sequence_ids)
        if sequence_id == 1
    ]

    best_candidate = {
        "text": "",
        "score": float("-inf")
    }

    for start_index in context_indices:
        max_end_index = min(
            start_index + max_answer_length,
            context_indices[-1] + 1
        )

        for end_index in range(start_index, max_end_index):
            if sequence_ids[end_index] != 1:
                continue

            start_character = offsets[start_index][0]
            end_character = offsets[end_index][1]

            answer_text = context[
                start_character:end_character
            ].strip()

            if not answer_text:
                continue

            score = (
                float(start_scores[start_index])
                + float(end_scores[end_index])
            )

            if score > best_candidate["score"]:
                best_candidate = {
                    "text": answer_text,
                    "score": score
                }

    return best_candidate

In [83]:
all_paragraph_qa_predictions_50 = []

for row_index in tqdm(range(50)):
    row = val_df.iloc[row_index]

    question = (
        clean_text(row["postText"])
        + " "
        + clean_text(row["targetTitle"])
    ).strip()

    candidates = []

    for paragraph_index, paragraph in enumerate(
        row["targetParagraphs"]
    ):
        candidate = extract_qa_candidate(
            question,
            str(paragraph)
        )

        candidate["paragraph_index"] = paragraph_index
        candidates.append(candidate)

    best_candidate = max(
        candidates,
        key=lambda item: item["score"]
    )

    all_paragraph_qa_predictions_50.append(
        best_candidate["text"]
    )

  0%|          | 0/50 [00:00<?, ?it/s]

In [84]:
qa_full_article_scores_50 = []

for row_index, prediction in enumerate(
    all_paragraph_qa_predictions_50
):
    reference = spoiler_list_to_text(
        val_df.iloc[row_index]["spoiler"]
    )

    score = meteor_score(
        [reference.split()],
        prediction.split()
    )

    qa_full_article_scores_50.append(score)

print(
    "All-paragraph QA METEOR on first 50:",
    np.mean(qa_full_article_scores_50)
)

for row_index in range(5):
    print(f"\nExample {row_index}")
    print(
        "Reference:",
        spoiler_list_to_text(
            val_df.iloc[row_index]["spoiler"]
        )
    )
    print(
        "Prediction:",
        all_paragraph_qa_predictions_50[row_index]
    )

All-paragraph QA METEOR on first 50: 0.34396942853486534

Example 0
Reference: some of the plot elements are so disturbing that they are making him feel sick
Prediction: too dark

Example 1
Reference: "intentionally" could transform a court case against Phoenix-area Sheriff Joe Arpaio from civil charges to a criminal prosecution
Prediction: "The Defendants’ unfair, partial, and inequitable application of discipline disproportionally (sic) damaged members of the Plaintiff class,"

Example 2
Reference: 20%
Prediction: 20%

Example 3
Reference: Alan Rickman & Rupert Grint CBGB
Prediction: The Dead Boys

Example 4
Reference: a man who swallowed a 64GB microSD card and then pooped it into a strainer
Prediction: pooped it into a strainer so he could maybe recover the drone footage he shot on it


In [85]:
all_paragraph_qa_predictions = []

for row_index in tqdm(range(len(val_df))):
    row = val_df.iloc[row_index]

    question = (
        clean_text(row["postText"])
        + " "
        + clean_text(row["targetTitle"])
    ).strip()

    candidates = []

    for paragraph_index, paragraph in enumerate(
        row["targetParagraphs"]
    ):
        candidate = extract_qa_candidate(
            question,
            str(paragraph)
        )

        candidate["paragraph_index"] = paragraph_index
        candidates.append(candidate)

    best_candidate = max(
        candidates,
        key=lambda item: item["score"]
    )

    all_paragraph_qa_predictions.append(
        best_candidate["text"]
    )

print("Predictions:", len(all_paragraph_qa_predictions))

  0%|          | 0/400 [00:00<?, ?it/s]

Predictions: 400


In [86]:
all_paragraph_qa_scores = []

for reference, prediction in zip(
    val_df["spoiler"],
    all_paragraph_qa_predictions
):
    reference_text = spoiler_list_to_text(reference)

    score = meteor_score(
        [reference_text.split()],
        prediction.split()
    )

    all_paragraph_qa_scores.append(score)

ALL_PARAGRAPH_QA_METEOR = float(
    np.mean(all_paragraph_qa_scores)
)

print("Previous predicted-type semantic METEOR:", PREDICTED_TYPE_METEOR)
print("All-paragraph QA METEOR:", ALL_PARAGRAPH_QA_METEOR)

Previous predicted-type semantic METEOR: 0.20499894609238228
All-paragraph QA METEOR: 0.3177886294532388


In [87]:

qa_full_results_df = pd.DataFrame({
    "spoiler_type": val_df["tags"].apply(get_spoiler_type),
    "reference": val_df["spoiler"].apply(spoiler_list_to_text),
    "prediction": all_paragraph_qa_predictions,
    "meteor": all_paragraph_qa_scores
})

print(
    qa_full_results_df.groupby("spoiler_type")["meteor"]
    .agg(["mean", "count"])
)

                  mean  count
spoiler_type                 
multi         0.184409     84
passage       0.215305    154
phrase        0.484371    162


In [88]:
comparison_df = pd.DataFrame({
    "spoiler_type": val_df["tags"].apply(get_spoiler_type),
    "reference": val_df["spoiler"].apply(spoiler_list_to_text),
    "qa_prediction": all_paragraph_qa_predictions,
    "semantic_prediction": predicted_type_val_predictions
})

comparison_df["qa_meteor"] = comparison_df.apply(
    lambda row: meteor_score(
        [row["reference"].split()],
        row["qa_prediction"].split()
    ),
    axis=1
)

comparison_df["semantic_meteor"] = comparison_df.apply(
    lambda row: meteor_score(
        [row["reference"].split()],
        row["semantic_prediction"].split()
    ),
    axis=1
)

print("QA performance by type:")
print(
    comparison_df.groupby("spoiler_type")["qa_meteor"]
    .mean()
)

print("\nSemantic performance by type:")
print(
    comparison_df.groupby("spoiler_type")["semantic_meteor"]
    .mean()
)

QA performance by type:
spoiler_type
multi      0.184409
passage    0.215305
phrase     0.484371
Name: qa_meteor, dtype: float64

Semantic performance by type:
spoiler_type
multi      0.168446
passage    0.316146
phrase     0.118294
Name: semantic_meteor, dtype: float64


In [91]:
hybrid_val_predictions = []

for row_index in range(len(val_df)):
    predicted_type = classifier_val_predicted_types[row_index]

    if predicted_type == "phrase":
        prediction = all_paragraph_qa_predictions[row_index]
    else:
        prediction = predicted_type_val_predictions[row_index]

    hybrid_val_predictions.append(prediction)

In [92]:
hybrid_meteor_scores = []

for reference, prediction in zip(
    val_df["spoiler"],
    hybrid_val_predictions
):
    reference_text = spoiler_list_to_text(reference)

    score = meteor_score(
        [reference_text.split()],
        prediction.split()
    )

    hybrid_meteor_scores.append(score)

HYBRID_METEOR = float(
    np.mean(hybrid_meteor_scores)
)

print("Semantic pipeline METEOR:", PREDICTED_TYPE_METEOR)
print("All-paragraph QA METEOR:", ALL_PARAGRAPH_QA_METEOR)
print("Hybrid METEOR:", HYBRID_METEOR)

Semantic pipeline METEOR: 0.20499894609238228
All-paragraph QA METEOR: 0.3177886294532388
Hybrid METEOR: 0.326209190496308


In [101]:
qa_submission = sample_solution.copy()

qa_submission["spoiler"] = [
    str(prediction).strip()
    for prediction in test_all_paragraph_qa_predictions
]

qa_submission_path = (
    "/kaggle/working/task2_all_paragraph_qa_submission.csv"
)

qa_submission.to_csv(
    qa_submission_path,
    index=False
)

print("Submission shape:", qa_submission.shape)
print("Columns:", qa_submission.columns.tolist())
print("Missing spoilers:", qa_submission["spoiler"].isna().sum())
print(
    "Empty spoilers:",
    (qa_submission["spoiler"].str.len() == 0).sum()
)

display(qa_submission.head())

print("Saved to:", qa_submission_path)

Submission shape: (400, 2)
Columns: ['id', 'spoiler']
Missing spoilers: 0
Empty spoilers: 0


,id,spoiler
0,0,Graham McMillan
1,1,1. Prioritise - say yes when it matters most.
2,2,higher taxes
3,3,Braconid
4,4,Cured egg yolks


Saved to: /kaggle/working/task2_all_paragraph_qa_submission.csv


In [103]:
semantic_submission = pd.read_csv(
    "/kaggle/working/task2_semantic_type_aware_submission.csv"
)

print("Semantic file shape:", semantic_submission.shape)
print("Semantic columns:", semantic_submission.columns.tolist())
display(semantic_submission.head())

Semantic file shape: (400, 2)
Semantic columns: ['id', 'spoiler']


,id,spoiler
0,0,"He has balloons and a sign in hand that reads,..."
1,1,And in the workplace putting your needs before...
2,2,But the ability of money and the things it buy...
3,3,"In other news, I’ve literally never won a game..."
4,4,Even better if you're raising chickens at home...


In [104]:
hybrid_test_predictions = []

for row_index in range(len(test_df)):
    predicted_type = classifier_test_predicted_types[row_index]

    if predicted_type == "phrase":
        prediction = test_all_paragraph_qa_predictions[row_index]
    else:
        prediction = semantic_submission.loc[row_index, "spoiler"]

    hybrid_test_predictions.append(str(prediction).strip())

In [105]:
hybrid_submission = sample_solution.copy()
hybrid_submission["spoiler"] = hybrid_test_predictions

hybrid_submission_path = (
    "/kaggle/working/task2_hybrid_qa_semantic_submission.csv"
)

hybrid_submission.to_csv(
    hybrid_submission_path,
    index=False
)

print("Submission shape:", hybrid_submission.shape)
print("Missing:", hybrid_submission["spoiler"].isna().sum())
print("Empty:", (hybrid_submission["spoiler"].str.len() == 0).sum())

print("\nPredictions selected from each approach:")
print(pd.Series(classifier_test_predicted_types).value_counts())

display(hybrid_submission.head())
print("Saved to:", hybrid_submission_path)

Submission shape: (400, 2)
Missing: 0
Empty: 0

Predictions selected from each approach:
passage    188
phrase     160
multi       52
Name: count, dtype: int64


,id,spoiler
0,0,Graham McMillan
1,1,And in the workplace putting your needs before...
2,2,higher taxes
3,3,Braconid
4,4,Even better if you're raising chickens at home...


Saved to: /kaggle/working/task2_hybrid_qa_semantic_submission.csv


In [106]:
import re
import numpy as np


def expand_qa_answer_to_sentence(answer, paragraphs):
    answer = str(answer).strip()

    if not answer:
        return answer

    for paragraph in paragraphs:
        paragraph = str(paragraph).strip()

        if not paragraph:
            continue

        answer_position = paragraph.lower().find(answer.lower())

        if answer_position == -1:
            continue

        # Simple sentence splitting without requiring NLTK downloads
        sentence_matches = list(
            re.finditer(
                r'[^.!?]+(?:[.!?]+|$)',
                paragraph
            )
        )

        for sentence_index, match in enumerate(sentence_matches):
            sentence_start = match.start()
            sentence_end = match.end()

            if sentence_start <= answer_position < sentence_end:
                selected_text = match.group().strip()

                # Add the next sentence when the selected sentence is short
                if (
                    len(selected_text.split()) < 12
                    and sentence_index + 1 < len(sentence_matches)
                ):
                    next_sentence = (
                        sentence_matches[sentence_index + 1]
                        .group()
                        .strip()
                    )

                    selected_text = (
                        selected_text + " " + next_sentence
                    ).strip()

                return selected_text

    return answer


expanded_val_predictions = []

for row_index in range(len(val_df)):
    qa_prediction = all_paragraph_qa_predictions[row_index]
    predicted_type = classifier_val_predicted_types[row_index]

    if predicted_type == "passage":
        prediction = expand_qa_answer_to_sentence(
            qa_prediction,
            val_df.iloc[row_index]["targetParagraphs"]
        )
    else:
        prediction = qa_prediction

    expanded_val_predictions.append(prediction)


expanded_scores = []

for reference, prediction in zip(
    val_df["spoiler"],
    expanded_val_predictions
):
    reference_text = spoiler_list_to_text(reference)

    expanded_scores.append(
        meteor_score(
            [reference_text.split()],
            str(prediction).split()
        )
    )


EXPANDED_QA_METEOR = float(np.mean(expanded_scores))

print("Original QA METEOR:", ALL_PARAGRAPH_QA_METEOR)
print("Passage-expanded QA METEOR:", EXPANDED_QA_METEOR)

Original QA METEOR: 0.3177886294532388
Passage-expanded QA METEOR: 0.32390279059624905


In [107]:
multi_span_val_predictions = []

for row_index in tqdm(range(len(val_df))):
    row = val_df.iloc[row_index]

    question = (
        clean_text(row["postText"])
        + " "
        + clean_text(row["targetTitle"])
    ).strip()

    candidates = []

    for paragraph_index, paragraph in enumerate(
        row["targetParagraphs"]
    ):
        candidate = extract_qa_candidate(
            question,
            str(paragraph)
        )

        candidate["paragraph_index"] = paragraph_index

        if candidate["text"].strip():
            candidates.append(candidate)

    candidates = sorted(
        candidates,
        key=lambda item: item["score"],
        reverse=True
    )

    predicted_type = classifier_val_predicted_types[row_index]

    if predicted_type == "phrase":
        prediction = (
            candidates[0]["text"]
            if candidates
            else ""
        )

    elif predicted_type == "passage":
        base_answer = (
            candidates[0]["text"]
            if candidates
            else ""
        )

        prediction = expand_qa_answer_to_sentence(
            base_answer,
            row["targetParagraphs"]
        )

    else:  # multi
        selected_answers = []
        used_paragraphs = set()

        for candidate in candidates:
            paragraph_index = candidate["paragraph_index"]
            answer_text = candidate["text"].strip()

            if paragraph_index in used_paragraphs:
                continue

            if answer_text in selected_answers:
                continue

            selected_answers.append(answer_text)
            used_paragraphs.add(paragraph_index)

            if len(selected_answers) == 3:
                break

        prediction = " ".join(selected_answers)

    multi_span_val_predictions.append(prediction)

  0%|          | 0/400 [00:00<?, ?it/s]

In [108]:
multi_span_scores = []

for reference, prediction in zip(
    val_df["spoiler"],
    multi_span_val_predictions
):
    reference_text = spoiler_list_to_text(reference)

    multi_span_scores.append(
        meteor_score(
            [reference_text.split()],
            prediction.split()
        )
    )

MULTI_SPAN_QA_METEOR = float(
    np.mean(multi_span_scores)
)

print("QA-only METEOR:", ALL_PARAGRAPH_QA_METEOR)
print("Passage-expanded METEOR:", EXPANDED_QA_METEOR)
print("Multi-span QA METEOR:", MULTI_SPAN_QA_METEOR)

QA-only METEOR: 0.3177886294532388
Passage-expanded METEOR: 0.32390279059624905
Multi-span QA METEOR: 0.3480546939560489


In [109]:
multi_span_test_predictions = []

for row_index in tqdm(range(len(test_df))):
    row = test_df.iloc[row_index]

    question = (
        clean_text(row["postText"])
        + " "
        + clean_text(row["targetTitle"])
    ).strip()

    candidates = []

    for paragraph_index, paragraph in enumerate(
        row["targetParagraphs"]
    ):
        candidate = extract_qa_candidate(
            question,
            str(paragraph)
        )

        candidate["paragraph_index"] = paragraph_index

        if candidate["text"].strip():
            candidates.append(candidate)

    candidates = sorted(
        candidates,
        key=lambda item: item["score"],
        reverse=True
    )

    predicted_type = classifier_test_predicted_types[row_index]

    if predicted_type == "phrase":
        prediction = candidates[0]["text"] if candidates else ""

    elif predicted_type == "passage":
        base_answer = candidates[0]["text"] if candidates else ""

        prediction = expand_qa_answer_to_sentence(
            base_answer,
            row["targetParagraphs"]
        )

    else:  # multi
        selected_answers = []
        used_paragraphs = set()

        for candidate in candidates:
            paragraph_index = candidate["paragraph_index"]
            answer_text = candidate["text"].strip()

            if paragraph_index in used_paragraphs:
                continue

            if answer_text in selected_answers:
                continue

            selected_answers.append(answer_text)
            used_paragraphs.add(paragraph_index)

            if len(selected_answers) == 3:
                break

        prediction = " ".join(selected_answers)

    multi_span_test_predictions.append(prediction)

print("Multi-span test predictions:", len(multi_span_test_predictions))

  0%|          | 0/400 [00:00<?, ?it/s]

Multi-span test predictions: 400


In [110]:
multi_span_submission = sample_solution.copy()

multi_span_submission["spoiler"] = [
    str(prediction).strip()
    for prediction in multi_span_test_predictions
]

multi_span_submission_path = (
    "/kaggle/working/task2_multi_span_qa_submission.csv"
)

multi_span_submission.to_csv(
    multi_span_submission_path,
    index=False
)

print("Shape:", multi_span_submission.shape)
print("Missing:", multi_span_submission["spoiler"].isna().sum())
print("Empty:", (multi_span_submission["spoiler"].str.len() == 0).sum())

display(multi_span_submission.head())

print("Saved to:", multi_span_submission_path)

Shape: (400, 2)
Missing: 0
Empty: 0


,id,spoiler
0,0,Graham McMillan
1,1,1. Prioritise - say yes when it matters most.
2,2,higher taxes
3,3,Braconid
4,4,Cured egg yolks are delicious—but strong. Beca...


Saved to: /kaggle/working/task2_multi_span_qa_submission.csv


In [111]:
import random
import pandas as pd
from tqdm.auto import tqdm

random.seed(42)


def normalize_for_matching(text):
    return " ".join(str(text).lower().split())


def find_positive_paragraph_indices(spoiler_parts, paragraphs):
    positive_indices = set()

    normalized_paragraphs = [
        normalize_for_matching(paragraph)
        for paragraph in paragraphs
    ]

    for spoiler_part in spoiler_parts:
        normalized_spoiler = normalize_for_matching(spoiler_part)

        if not normalized_spoiler:
            continue

        for paragraph_index, normalized_paragraph in enumerate(
            normalized_paragraphs
        ):
            if normalized_spoiler in normalized_paragraph:
                positive_indices.add(paragraph_index)
                break

    return sorted(positive_indices)


def build_paragraph_ranking_pairs(
    dataframe,
    negatives_per_positive=3
):
    ranking_rows = []

    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe)
    ):
        paragraphs = [
            str(paragraph).strip()
            for paragraph in row["targetParagraphs"]
        ]

        positive_indices = find_positive_paragraph_indices(
            row["spoiler"],
            paragraphs
        )

        if not positive_indices:
            continue

        query = (
            clean_text(row["postText"])
            + " "
            + clean_text(row["targetTitle"])
        ).strip()

        negative_indices = [
            index
            for index in range(len(paragraphs))
            if index not in positive_indices
            and paragraphs[index]
        ]

        for positive_index in positive_indices:
            ranking_rows.append({
                "query": query,
                "paragraph": paragraphs[positive_index],
                "label": 1
            })

            selected_negative_indices = random.sample(
                negative_indices,
                k=min(
                    negatives_per_positive,
                    len(negative_indices)
                )
            )

            for negative_index in selected_negative_indices:
                ranking_rows.append({
                    "query": query,
                    "paragraph": paragraphs[negative_index],
                    "label": 0
                })

    return pd.DataFrame(ranking_rows)


ranker_train_df = build_paragraph_ranking_pairs(
    train_df,
    negatives_per_positive=3
)

ranker_val_df = build_paragraph_ranking_pairs(
    val_df,
    negatives_per_positive=3
)

print("Ranker training shape:", ranker_train_df.shape)
print("Ranker validation shape:", ranker_val_df.shape)

print("\nTraining label counts:")
print(ranker_train_df["label"].value_counts())

print("\nValidation label counts:")
print(ranker_val_df["label"].value_counts())

display(ranker_train_df.sample(5, random_state=42))

  0%|          | 0/3200 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

Ranker training shape: (16830, 3)
Ranker validation shape: (2248, 3)

Training label counts:
label
0    12511
1     4319
Name: count, dtype: int64

Validation label counts:
label
0    1673
1     575
Name: count, dtype: int64


,query,paragraph,label
16276,Could You Be A Barista? Could You Be A Barista?,"A ""corretto"", also known as an ""espresso corre...",1
8899,You Won’t Believe How Many Copies Grand Theft ...,"""Grand Theft Auto V has shipped 65 million cop...",1
3013,You'll never guess what's inside this tent You...,"The tents’ interior looks like a classroom, an...",1
11354,Bigg Boss 10: You won't believe how much Karan...,A lot of cajoling had to happen to get Karan t...,0
13798,"New Zealand town with ""too many jobs"" will giv...","The town of about 800 is located on a lush, gr...",0


In [112]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

RANKER_MODEL_NAME = "roberta-base"
RANKER_MAX_LENGTH = 256

ranker_tokenizer = AutoTokenizer.from_pretrained(
    RANKER_MODEL_NAME
)

ranker_model = AutoModelForSequenceClassification.from_pretrained(
    RANKER_MODEL_NAME,
    num_labels=2
)

ranker_train_dataset = Dataset.from_pandas(
    ranker_train_df,
    preserve_index=False
)

ranker_val_dataset = Dataset.from_pandas(
    ranker_val_df,
    preserve_index=False
)


def tokenize_ranker_batch(examples):
    return ranker_tokenizer(
        examples["query"],
        examples["paragraph"],
        truncation=True,
        max_length=RANKER_MAX_LENGTH
    )


ranker_train_tokenized = ranker_train_dataset.map(
    tokenize_ranker_batch,
    batched=True,
    remove_columns=["query", "paragraph"]
)

ranker_val_tokenized = ranker_val_dataset.map(
    tokenize_ranker_batch,
    batched=True,
    remove_columns=["query", "paragraph"]
)

ranker_data_collator = DataCollatorWithPadding(
    tokenizer=ranker_tokenizer
)


def compute_ranker_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


ranker_training_args = TrainingArguments(
    output_dir="/kaggle/working/task2_paragraph_ranker",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42
)

ranker_trainer = Trainer(
    model=ranker_model,
    args=ranker_training_args,
    train_dataset=ranker_train_tokenized,
    eval_dataset=ranker_val_tokenized,
    processing_class=ranker_tokenizer,
    data_collator=ranker_data_collator,
    compute_metrics=compute_ranker_metrics
)

ranker_trainer.train()

ranker_eval_results = ranker_trainer.evaluate()
print(ranker_eval_results)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/16830 [00:00<?, ? examples/s]

Map:   0%|          | 0/2248 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': 0.8866483569145203, 'eval_accuracy': 0.806049822064057, 'eval_precision': 0.640973630831643, 'eval_recall': 0.5495652173913044, 'eval_f1': 0.5917602996254682, 'eval_runtime': 15.541, 'eval_samples_per_second': 144.65, 'eval_steps_per_second': 2.316, 'epoch': 2.0}


In [113]:
def rank_paragraphs_with_trained_ranker(row):
    query = (
        clean_text(row["postText"])
        + " "
        + clean_text(row["targetTitle"])
    ).strip()

    paragraphs = [
        str(paragraph).strip()
        for paragraph in row["targetParagraphs"]
    ]

    valid_indices = [
        index
        for index, paragraph in enumerate(paragraphs)
        if paragraph
    ]

    if not valid_indices:
        return []

    valid_paragraphs = [
        paragraphs[index]
        for index in valid_indices
    ]

    inputs = ranker_tokenizer(
        [query] * len(valid_paragraphs),
        valid_paragraphs,
        truncation=True,
        max_length=RANKER_MAX_LENGTH,
        padding=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(ranker_model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        logits = ranker_model(**inputs).logits

    positive_scores = torch.softmax(
        logits,
        dim=1
    )[:, 1].cpu().numpy()

    ranked_local_indices = np.argsort(
        positive_scores
    )[::-1]

    return [
        valid_indices[int(local_index)]
        for local_index in ranked_local_indices
    ]

In [114]:
top1_hits = 0
top3_hits = 0
top5_hits = 0
total_parts = 0

for _, row in tqdm(
    val_df.iterrows(),
    total=len(val_df)
):
    ranked_indices = rank_paragraphs_with_trained_ranker(row)

    positive_indices = find_positive_paragraph_indices(
        row["spoiler"],
        row["targetParagraphs"]
    )

    for positive_index in positive_indices:
        total_parts += 1

        if positive_index in ranked_indices[:1]:
            top1_hits += 1

        if positive_index in ranked_indices[:3]:
            top3_hits += 1

        if positive_index in ranked_indices[:5]:
            top5_hits += 1

print("Matched validation spoiler parts:", total_parts)
print("Ranker top-1 recall:", top1_hits / total_parts)
print("Ranker top-3 recall:", top3_hits / total_parts)
print("Ranker top-5 recall:", top5_hits / total_parts)

  0%|          | 0/400 [00:00<?, ?it/s]

Matched validation spoiler parts: 575
Ranker top-1 recall: 0.3356521739130435
Ranker top-3 recall: 0.64
Ranker top-5 recall: 0.808695652173913


In [115]:
ranker_qa_val_predictions = []

for row_index in tqdm(range(len(val_df))):
    row = val_df.iloc[row_index]

    question = (
        clean_text(row["postText"])
        + " "
        + clean_text(row["targetTitle"])
    ).strip()

    ranked_paragraph_indices = rank_paragraphs_with_trained_ranker(row)
    selected_indices = ranked_paragraph_indices[:5]

    candidates = []

    for paragraph_index in selected_indices:
        paragraph = str(
            row["targetParagraphs"][paragraph_index]
        ).strip()

        if not paragraph:
            continue

        candidate = extract_qa_candidate(
            question,
            paragraph
        )

        candidate["paragraph_index"] = paragraph_index

        if candidate["text"].strip():
            candidates.append(candidate)

    candidates = sorted(
        candidates,
        key=lambda item: item["score"],
        reverse=True
    )

    predicted_type = classifier_val_predicted_types[row_index]

    if predicted_type == "phrase":
        prediction = candidates[0]["text"] if candidates else ""

    elif predicted_type == "passage":
        base_answer = candidates[0]["text"] if candidates else ""

        prediction = expand_qa_answer_to_sentence(
            base_answer,
            row["targetParagraphs"]
        )

    else:
        selected_answers = []
        used_paragraphs = set()

        for candidate in candidates:
            paragraph_index = candidate["paragraph_index"]
            answer_text = candidate["text"].strip()

            if paragraph_index in used_paragraphs:
                continue

            if answer_text in selected_answers:
                continue

            selected_answers.append(answer_text)
            used_paragraphs.add(paragraph_index)

            if len(selected_answers) == 3:
                break

        prediction = " ".join(selected_answers)

    ranker_qa_val_predictions.append(prediction)

  0%|          | 0/400 [00:00<?, ?it/s]

In [116]:
ranker_qa_scores = []

for reference, prediction in zip(
    val_df["spoiler"],
    ranker_qa_val_predictions
):
    reference_text = spoiler_list_to_text(reference)

    ranker_qa_scores.append(
        meteor_score(
            [reference_text.split()],
            str(prediction).split()
        )
    )

RANKER_QA_METEOR = float(np.mean(ranker_qa_scores))

print("All-paragraph multi-span QA:", MULTI_SPAN_QA_METEOR)
print("Ranker top-5 multi-span QA:", RANKER_QA_METEOR)

All-paragraph multi-span QA: 0.3480546939560489
Ranker top-5 multi-span QA: 0.3636887763226319


In [117]:
ranker_qa_test_predictions = []

for row_index in tqdm(range(len(test_df))):
    row = test_df.iloc[row_index]

    question = (
        clean_text(row["postText"])
        + " "
        + clean_text(row["targetTitle"])
    ).strip()

    ranked_paragraph_indices = rank_paragraphs_with_trained_ranker(row)
    selected_indices = ranked_paragraph_indices[:5]

    candidates = []

    for paragraph_index in selected_indices:
        paragraph = str(
            row["targetParagraphs"][paragraph_index]
        ).strip()

        if not paragraph:
            continue

        candidate = extract_qa_candidate(
            question,
            paragraph
        )

        candidate["paragraph_index"] = paragraph_index

        if candidate["text"].strip():
            candidates.append(candidate)

    candidates = sorted(
        candidates,
        key=lambda item: item["score"],
        reverse=True
    )

    predicted_type = classifier_test_predicted_types[row_index]

    if predicted_type == "phrase":
        prediction = candidates[0]["text"] if candidates else ""

    elif predicted_type == "passage":
        base_answer = candidates[0]["text"] if candidates else ""

        prediction = expand_qa_answer_to_sentence(
            base_answer,
            row["targetParagraphs"]
        )

    else:  # multi
        selected_answers = []
        used_paragraphs = set()

        for candidate in candidates:
            paragraph_index = candidate["paragraph_index"]
            answer_text = candidate["text"].strip()

            if paragraph_index in used_paragraphs:
                continue

            if answer_text in selected_answers:
                continue

            selected_answers.append(answer_text)
            used_paragraphs.add(paragraph_index)

            if len(selected_answers) == 3:
                break

        prediction = " ".join(selected_answers)

    ranker_qa_test_predictions.append(prediction)

print("Ranker-QA test predictions:", len(ranker_qa_test_predictions))

  0%|          | 0/400 [00:00<?, ?it/s]

Ranker-QA test predictions: 400


In [118]:
ranker_qa_submission = sample_solution.copy()

ranker_qa_submission["spoiler"] = [
    str(prediction).strip()
    for prediction in ranker_qa_test_predictions
]

ranker_qa_submission_path = (
    "/kaggle/working/task2_ranker_top5_qa_submission.csv"
)

ranker_qa_submission.to_csv(
    ranker_qa_submission_path,
    index=False
)

print("Shape:", ranker_qa_submission.shape)
print("Missing:", ranker_qa_submission["spoiler"].isna().sum())
print("Empty:", (ranker_qa_submission["spoiler"].str.len() == 0).sum())

display(ranker_qa_submission.head())

print("Saved to:", ranker_qa_submission_path)

Shape: (400, 2)
Missing: 0
Empty: 0


,id,spoiler
0,0,Graham McMillan
1,1,1. Prioritise - say yes when it matters most.
2,2,higher taxes
3,3,Braconid
4,4,4. Place the yolks on a paper towel to drain a...


Saved to: /kaggle/working/task2_ranker_top5_qa_submission.csv
